In [5]:
import numpy as np
import matplotlib.pyplot as plt 
from scipy.stats import binom 
from scipy.stats import poisson

'''The following functions are intended to simulate the SNP genotypes of two individuals and combine them at different mixture proportions
using read counts as a quantitative measurement'''

#Setting the parameters 
af = 0.5 # Allele frequency set to 0.5 given an equal probability of inhereting an allele from each parent
nsites = 10

#Function to simulate genotypes
def create_genotypes (af, nsites):
    '''From binomal distributions, create haplotypes and genotypes for two individuals'''

    # Simulating alleles for contributor 1 
    c1_hap1 = binom.rvs(1, af, size=nsites) # Simulating alleles for contributor 1 (haplotype 1) 
    c1_hap2 = binom.rvs(1, af, size=nsites) # Simulating alleles for contributor 1 (haplotype 2) 

    # Simulating alleles for contributor 2
    c2_hap1 = binom.rvs(1, af, size=nsites) # Simulating alleles for contributor 2 (haplotype 1) 
    c2_hap2 = binom.rvs(1, af, size=nsites) # Simulating alleles for contributor 2 (haplotype 2) 

    # Using simulated haplotypes to create genotypes for both contributors
    c1_geno = c1_hap1 + c1_hap2 # Add haplotypes to simulate contributor 1 genotype  
    c2_geno = c2_hap1 + c2_hap2 # Add haplotypes to simulate contributor 2 genotype 
    
    return c1_geno, c2_geno

#Function to calculate allele frequencies per contributor 
def calc_allele_freq (c_genotype): 
    c_allele_freq = c_genotype/2 #converts genotype (0,1,2) into allele frequency of allele 1 
    return c_allele_freq

#Simulate mixture data from simulated genotypes (and their frequencies) using a pre-determiend mixture proportion at given sequence depth 
def read_counts_from_genotypes(c1_allele_freq,c2_allele_freq,mix_prop,c1_exp_seq_depth,c2_exp_seq_depth):
    '''from two individuals, create a composite allele frequency and simulate reads for that mixture 
    under the mixing parameter and expected sequence depth.
    '''
    # number of sites needs to be consistent
    assert len(c1_allele_freq)==len(c2_allele_freq)
    nsites=len(c1_allele_freq)
    
    # create allele frequency of read pool based on mixing parameter
    comb_frequency = ((mix_prop*c1_allele_freq) + (1-mix_prop)*c2_allele_freq)
    
    # assumes a poisson distribution for the read coverage per site, parameterized by expected sequence depth for each contributor
    c1_seq_depth = poisson.rvs(c1_exp_seq_depth,size=nsites)
    c2_seq_depth = poisson.rvs(c2_exp_seq_depth, size=nsites) 
    seq_depth = c1_seq_depth + c2_seq_depth

    # Ensure combined sequence depth for two contributors is an integer 
    seq_depth = np.round(seq_depth).astype(int)
    
    # simulate counts of Allele 1 (indexed by the allele frequency, then use the overall counts to get # 0 alleles at the same sites
    simulate_counts1 = binom.rvs(seq_depth, comb_frequency, size = nsites)
    
    # print(simulate_counts1)
    simulate_counts0=(seq_depth-simulate_counts1)
    # print(simulate_counts0)
    return (simulate_counts0,simulate_counts1)

In [7]:
c1_geno, c2_geno = create_genotypes(af,nsites)
print(c1_geno)
print(c2_geno)

[1 1 2 0 0 0 1 0 1 1]
[1 1 0 1 1 0 1 1 1 2]


In [11]:
c1_allele_freq = calc_allele_freq(c1_geno)
c2_allele_freq = calc_allele_freq(c2_geno)
print(c1_allele_freq)
print(c2_allele_freq)

[0.5 0.5 1.  0.  0.  0.  0.5 0.  0.5 0.5]
[0.5 0.5 0.  0.5 0.5 0.  0.5 0.5 0.5 1. ]


In [17]:
totalallele0, totalallele1 = read_counts_from_genotypes(c1_allele_freq, c2_allele_freq, mix_prop=0.5, c1_exp_seq_depth=100, c2_exp_seq_depth=100)

print(totalallele0)
print(totalallele1)

[105  91 113 143 156 209 104 182  99  48]
[ 91 103  88  55  55   0 100  47 107 152]
